# Aufgabenspezifisches Encoder-Modell 

## Daten laden

Unabhängig vom verwendeten Modell laden wir zunächst die bereits installierten Pakete sowie unsere Textdaten und speichern diese in einem Listenobjekt:

In [8]:
#| echo: false
from transformers import logging
logging.set_verbosity_error()  # Mitteilungen muten, damit Dokuemnt ordentlich bleibt

In [9]:
import pandas as pd
from transformers import pipeline

daten = pd.read_csv("data/example_dataset.csv")
datenliste = list(daten["text"])

daten

,text
0,Die orthodoxe Gemeinde feiert heute #Ostern. I...
1,Vielen Dank Gregor Rutz für die Unterstützung ...
2,Wird ja immer schlimmer mit den Intoleranten d...
3,"Liebe Junge, geht wählen. Kann ja nicht sein, ..."
4,‘— Das war unser EU-Wahl-Abschluss der SP– mit...
5,Aufschlussreicher Blick hinter die Mauern! Tol...
6,"Denn ein Hard-Brexit würde der #EU, aber noch ..."
7,Familienbonus: Es ist erstaunlich wie viele Me...
8,Integration passiert vor Ort. Deshalb arbeitet...
9,Ich frage mich: Wer ist als nächstes dran Frau...


## Klassifikation

Im Gegensatz zu kategoriespezifischen Modellen können wir mit aufgabenspezifischen Encoder-Modellen jede beliebige Kategorie codieren. Als Beispiel verwenden wir hier das NLI-Modell von Laurer et al. (2024b; `model = "MoritzLaurer/bge-m3-zeroshot-v2.0"`). Für die Klassifikation mit dem NLI-Modell spezifizieren wir zusätzlich zum Modell noch die Ausprägungen der Kategorie (*`candidate_labels`) und eine “Hypothese”, die auf Grundlage des Textes geprüft werden soll (`hypothesis_template`; zur Hypothese im Kontext von NLI s. Abschn. 2.1). Bei der Ausführung des `pipeline()`-Befehls werden die zu codierenden Ausprägungen der Kategorie automatisch anstelle der geschwungenen Klammern in die Hypothese eingesetzt. Auf diese Weise wird der Befehl für jede Ausprägung einzeln durchgeführt (hier für “positiv”, “negativ” und “neutral”). Wir erhalten also als Ergebnis einen Datensatz, der uns pro Text für jede spezifizierte Ausprägung einen Wahrscheinlichkeitswert ausgibt (in unserem Beispiel die Wahrscheinlichkeit für “positiv”, “negativ” und “neutral”). Über den Parameter multi_label können wir festlegen, ob prinzipiell mehrere Kategorien gleichzeitig in einem Text vorkommen können.

In [10]:
template = "Die Stimmung beziehungsweise Tonalität des Textes ist {}"
labels = ["positiv", "neutral", "negativ"]

zeroshot_classifier = pipeline("zero-shot-classification",
                                model = "MoritzLaurer/bge-m3-zeroshot-v2.0")

results = zeroshot_classifier(datenliste, candidate_labels=labels,
                               hypothesis_template=template, multi_label=False)

labels = set(label for item in results for label in item['labels'])
out_dict = {label: [] for label in labels}
out_dict['sequence'] = []

for item in results:
    out_dict['sequence'].append(item['sequence'])
    for label in labels:
        if label in item['labels']:
            out_dict[label].append(item['scores'][item['labels'].index(label)])
        else:
            out_dict[label].append(None)

pd.DataFrame(out_dict)

,positiv,neutral,negativ,sequence
0,0.707576,0.285608,0.006816,Die orthodoxe Gemeinde feiert heute #Ostern. I...
1,0.654405,0.342164,0.003431,Vielen Dank Gregor Rutz für die Unterstützung ...
2,0.010227,0.180488,0.809285,Wird ja immer schlimmer mit den Intoleranten d...
3,0.350793,0.261388,0.387819,"Liebe Junge, geht wählen. Kann ja nicht sein, ..."
4,0.440438,0.539635,0.019927,‘— Das war unser EU-Wahl-Abschluss der SP– mit...
5,0.985130,0.012621,0.002250,Aufschlussreicher Blick hinter die Mauern! Tol...
6,0.007550,0.201777,0.790673,"Denn ein Hard-Brexit würde der #EU, aber noch ..."
7,0.375643,0.567706,0.056651,Familienbonus: Es ist erstaunlich wie viele Me...
8,0.301832,0.689889,0.008279,Integration passiert vor Ort. Deshalb arbeitet...
9,0.044657,0.746518,0.208825,Ich frage mich: Wer ist als nächstes dran Frau...
